# Week 1: Introduction to Deep Learning - Homework

**ML2: Advanced Machine Learning**

**Estimated Time**: 1 hour

---

This homework combines programming exercises and knowledge-based questions to reinforce this week's concepts.

## Setup

Run this cell to import necessary libraries:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print('✓ Libraries imported successfully')

---
## Part 1: Programming Exercises (60%)

Complete the following programming tasks. Read each description carefully and implement the requested functionality.

### Exercise 1: Experiment: Observing Feature Learning

**Time**: 8 min

Run this code to visualize what happens when a network learns features automatically vs. using hand-crafted features. Observe the outputs and answer the reflection questions below.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Simulate a simple pattern recognition task
# Pattern: Detect if sum of inputs > 5
np.random.seed(42)
torch.manual_seed(42)

# Generate data
X = torch.randn(100, 4)  # 100 samples, 4 features
y = (X.sum(dim=1) > 0).float()  # Label: 1 if sum > 0, else 0

# Network that LEARNS features
model = nn.Sequential(
    nn.Linear(4, 8),   # Learned feature extraction
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

# Train for a few steps
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.BCELoss()

for epoch in range(20):
    optimizer.zero_grad()
    predictions = model(X).squeeze()
    loss = loss_fn(predictions, y)
    loss.backward()
    optimizer.step()

print(f"Final loss: {loss.item():.4f}")
print(f"First layer weights (learned features):")
print(model[0].weight.data)

# TODO: After running, answer reflection questions below

### Exercise 2: Experiment: Network Without Nonlinearity

**Time**: 10 min

This experiment demonstrates why activation functions are essential. Compare two networks: one with ReLU, one without.

In [ ]:
import torch
import torch.nn as nn

# Network WITH nonlinearity (ReLU)
network_with_relu = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 15),
    nn.ReLU(),
    nn.Linear(15, 5)
)

# Network WITHOUT nonlinearity (just linear layers)
network_without_relu = nn.Sequential(
    nn.Linear(10, 20),
    nn.Linear(20, 15),
    nn.Linear(15, 5)
)

# Test input
x = torch.randn(1, 10)

# Compare outputs
output_with = network_with_relu(x)
output_without = network_without_relu(x)

print("With ReLU output:", output_with)
print("Without ReLU output:", output_without)

# TODO: Now manually compute what network_without_relu is equivalent to
# Hint: Multiple linear transformations collapse into a single linear transformation
# Can you express the 3-layer linear network as a SINGLE equivalent linear layer?

In [ ]:
# Show that 3 linear layers without activation collapse to 1 linear layer

W1 = network_without_relu[0].weight.data  # shape: (20, 10)
W2 = network_without_relu[1].weight.data  # shape: (15, 20)
W3 = network_without_relu[2].weight.data  # shape:  (5, 15)

# Matrix multiplication is associative: this gives one equivalent matrix
W_star = W3 @ W2 @ W1  # shape: (5, 10)

print(f"W1: {W1.shape},  W2: {W2.shape},  W3: {W3.shape}")
print(f"W* = W3 @ W2 @ W1: {W_star.shape}")
print()
print("A single nn.Linear(10, 5) is mathematically equivalent to all three layers.")
print("Without activation functions, depth adds no representational power.")


---
## Part 2: Knowledge Questions (40%)

Answer the following questions to test your conceptual understanding.

### Question 1 (Short Answer)

**Question 1 - Automatic Feature Learning (Conceptual)**

Traditional machine learning for image classification requires manually designing features (e.g., edge detectors, color histograms, texture filters). Deep learning does not.

Explain in 3-4 sentences:
1. WHY can deep networks learn features automatically?
2. WHAT enables this (what architectural property)?
3. What is the tradeoff (what does deep learning need more of)?

**Hint**: Think about what happens in each layer of a deep network and how backpropagation adjusts those layers.

**Your Answer**:

Deep networks learn features automatically because every weight can be differentiated: backpropagation computes how much each weight contributed to the prediction error, and gradient descent adjusts them accordingly. Repeated across many examples, weights converge toward patterns that consistently reduce the loss. The tradeoff is data: the network needs enough labeled examples to learn meaningful features rather than memorizing noise.

### Question 2 (Short Answer)

**Question 2 - Feature Hierarchy (Conceptual)**

In a deep CNN for face recognition:
- Layer 1 might detect edges
- Layer 2 might detect facial features (eyes, nose)
- Layer 3 might detect whole faces

Explain: Why does depth create this hierarchy? What would happen if you used a single-layer network instead?

**Hint**: Consider how each layer builds on representations from the previous layer.

**Your Answer**:

Depth creates hierarchy because each layer's output becomes the next layer's input, Later layers see combinations of detected patterns rather than raw pixels, enabling progressively more abstract representations to emerge. Layer 1 detects edges, Layer 2 combines edges into shapes, Layer 3 combines shapes into objects. A single-layer network must map directly from raw inputs to class labels with no intermediate building blocks, severely limiting what it can represent.

### Question 3 (Short Answer)

**Question 3 - Nonlinearity Experiment Reflection**

Based on the 'Network Without Nonlinearity' experiment above:

Prove mathematically or explain conceptually why the 3-layer network without ReLU is equivalent to a SINGLE linear layer. What does this tell you about the necessity of activation functions?

**Hint**: Remember: Linear(Linear(x)) = Linear(x) because you can multiply weight matrices together.

**Your Answer**:

For three linear layers: h1 = W1·x, h2 = W2·h1, output = W3·h2. Substituting: output = W3·(W2·(W1·x)) = (W3·W2·W1)·x = W*·x — a single matrix multiplication. The code cell above confirms this: W* has shape (5, 10), the same effective input/output as all three layers combined. Without activation functions, depth is algebraically equivalent to a single linear layer and adds no representational power.

### Question 4 (Multiple Choice)

**Question 4 - Understanding Nonlinearity**

A neural network with 10 layers but NO activation functions can represent:

A) Any possible function (universal approximation)
B) Only linear functions
C) Only polynomial functions
D) Only step functions

A) Any possible function (universal approximation)
B) Only linear functions
C) Only polynomial functions
D) Only step functions

**Hint**: What happens when you compose linear transformations?

**Your Answer**: B

**Explanation**: Composing any number of linear transformations yields another linear transformation, it collapses into W3·(W2·(W1·x)) = W*·x. Regardless of depth, the network can only represent linear functions of its input. It draws flat hyperplanes through data, which is insufficient for virtually every real-world problem.

### Question 5 (Short Answer)

**Question 5 - ReLU Design Choice**

ReLU(x) = max(0, x) is one of the simplest possible nonlinear functions. Yet it became the dominant activation function (replacing sigmoid).

Explain TWO advantages ReLU has over sigmoid for deep networks. One should relate to gradients, one to computation.

**Hint**: Think about what happens to gradients when x is large and positive in sigmoid vs ReLU.

**Your Answer**:

Gradient advantage: Sigmoid saturates for large positive or negative inputs. Its gradient approaches zero, causing the vanishing gradient problem where the error signal nearly disappears as it propagates back through deep networks. ReLU's gradient is 1 for all positive inputs, so the signal passes backward without shrinking.

Computational advantage: ReLU is `max(0, x)` = one comparison operation. Sigmoid requires computing `1 / (1 + e^(-x))`, involving an exponential that is significantly more expensive across millions of neurons over thousands of training steps.

### Question 6 (Short Answer)

**Question 6 - Gradient Descent Intuition**

Gradient descent updates weights using: θ_new = θ_old - α × ∇L

Where ∇L is the gradient of the loss.

Explain in simple terms:
1. What does the gradient ∇L represent geometrically?
2. Why do we SUBTRACT it (the negative sign)?
3. What role does α (learning rate) play?

**Hint**: Think of the loss function as a landscape/terrain you're trying to navigate.

**Your Answer**:

The gradient ∇L is the direction of steepest ascent on the loss surface. It points toward higher loss. We subtract it because we want to move in the opposite direction, downhill toward lower loss. The learning rate α controls step size: too large and we overshoot the minimum; too small and convergence is very slow. In landscape terms: ∇L says which way is uphill, subtraction means we go downhill, and α determines how far we step.

### Question 7 (Multiple Choice)

**Question 7 - Loss Function Purpose**

The loss function in deep learning serves to:

A) Measure how wrong the model is, providing a signal for gradient descent
B) Prevent overfitting by penalizing complex models
C) Speed up training by reducing computation
D) Automatically select which features to learn

A) Measure how wrong the model is, providing a signal for gradient descent
B) Prevent overfitting by penalizing complex models
C) Speed up training by reducing computation
D) Automatically select which features to learn

**Hint**: What do we need to compute gradients?

**Your Answer**: A

**Explanation**: The loss function quantifies prediction error as a single differentiable number. Backpropagation differentiates that number with respect to every weight in the network. Without a loss there is no gradient and gradient descent has nothing to optimize. Option B describes regularization, a separate concept that can be added on top of the loss.

### Question 8 (Short Answer)

**Question 8 - Feature Learning Reflection**

After running the 'Observing Feature Learning' experiment:

Look at the learned weights in the first layer. These represent the FEATURES the network learned.

Explain: How did the network 'know' which features to learn? What guided it to learn useful features rather than random ones?

**Hint**: The answer involves both the loss function and backpropagation.

**Your Answer**:

The network did not know in advance, it was guided by the loss function and backpropagation. Each training step, the loss measured how wrong the prediction was, and backprop propagated that error signal back to the first layer. Gradient descent then nudged the weights in whichever direction reduced the loss. Over 20 epochs, weights that detected patterns predictive of "sum > 0" were reinforced, but weights that did not help were not. The loss is what gives the learning signal its direction.

### Question 9 (Short Answer)

**Question 9 - Connecting the Concepts**

Integrate all three key insights:

Explain how (1) automatic feature learning, (2) nonlinearity, and (3) gradient descent work TOGETHER to enable deep learning.

Your answer should show how all three are necessary and how they interact.

**Hint**: Think: What would happen if you removed any one of these three components?

**Your Answer**:

Automatic feature learning means weights adjust to detect useful patterns, but the network needs a signal defining what "useful" means. The loss function provides that signal, and backpropagation efficiently delivers it to every weight. Nonlinearity makes depth meaningful: without activation functions, all layers collapse to one linear transformation and gradient descent cannot learn nonlinear structure. 

### Question 10 (Short Answer)

**Question 10 - Scaling to Real Problems**

ImageNet (image classification) has 1000 classes and ~1.2 million training images. Traditional ML would require human experts to manually design thousands of features.

Explain: Why does deep learning have an advantage that GROWS as the problem gets more complex (more classes, more data)? What breaks down in the traditional approach?

**Hint**: Consider both the human effort required and what happens when you have more data.

**Your Answer**:

Hand-engineered features are bottlenecked by human expertise. As problems scale to 1000 classes across millions of images, no team can anticipate every relevant visual pattern. Deep learning's advantage compounds with scale because more data gives gradient descent a stronger, more diverse signal, enabling the network to automatically discover increasingly fine-grained features. The traditional pipeline breaks down because the human design step becomes the constraint, not data or compute.

---
## Submission

Before submitting:
1. Run all cells to ensure code executes without errors
2. Check that all questions are answered
3. Review your explanations for clarity

**To Submit**:
- File → Download → Download .ipynb
- Submit the notebook file to your course LMS

**Note**: Make sure your name is in the filename (e.g., homework_01_yourname.ipynb)